___
# <center>Árvores de decisão, florestas aleatórias e o arquivo de predições</center>
___

## Aula 24

**Objetivo.** Ajustar e ler uma árvore, controlar a profundidade, ajustar uma
floresta, comparar com a regressão logística da aula 23 e gerar o arquivo de
predições. A parte 5 é a **receita do Projeto 08**.

___
<div id="indice"></div>

## Índice

- [A base](#base)
- [1. Uma árvore de decisão](#parte1)
- [2. Escolher a profundidade](#parte2)
- [3. Floresta aleatória](#parte3)
- [4. Comparar os modelos](#parte4)
- [5. O arquivo de predições: a receita do Projeto 08](#parte5)

___
<div id="base"></div>

# A base

A mesma da aula 23: acórdãos de recursos criminais do TJSP, com a pergunta
**houve voto divergente?**. A célula abaixo refaz tudo da aula 23: base, preparo,
separação entre treino e teste, e as pastas da validação cruzada.

In [ ]:
import numpy as np
import pandas as pd
from plotnine import *
from sklearn.compose import make_column_transformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree

base = pd.read_csv("https://jtrecenti.github.io/cdad2-202662/ex08-competicao/dados/treino.csv")
base = base.dropna(subset=["unanimidade"])
base["divergiu"] = (base["unanimidade"] == "Maioria").astype(int)
print(base.shape)
base.head(3)

categoricas = ["assunto", "camara", "comarca", "polo_mp", "rel_faculdade", "rel_tipo", "rel_quinto"]
numericas = ["ano_publicacao", "tempo", "rel_idade", "rel_tempo_magistratura"]

def cria_preparo():
    """Dummies nas categóricas e padronização nas numéricas, com faltantes preenchidos."""
    return make_column_transformer(
        (make_pipeline(SimpleImputer(strategy="most_frequent"),
                       OneHotEncoder(handle_unknown="ignore", min_frequency=5)), categoricas),
        (make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), numericas),
    )

colunas = categoricas + numericas
X_treino, X_teste, y_treino, y_teste = train_test_split(
    base[colunas], base["divergiu"], test_size=0.25, stratify=base["divergiu"], random_state=23)
pastas = StratifiedKFold(5, shuffle=True, random_state=23)

[Volta ao Índice](#indice)

___
<div id="parte1"></div>

# 1. Uma árvore de decisão

A árvore usa o mesmo preparo. `max_depth` limita quantas perguntas seguidas ela
faz. Complete com profundidade 2, para dar para ler.

In [ ]:
arvore = make_pipeline(cria_preparo(), DecisionTreeClassifier(max_depth=____, random_state=1))
arvore.fit(X_treino, y_treino)

nomes = arvore[0].get_feature_names_out()
nomes = [n.split("__")[-1] for n in nomes]
_ = plot_tree(arvore[1], feature_names=nomes, filled=True, impurity=False, proportion=True, fontsize=8)

Em cada caixa, `value` traz a proporção de acórdãos unânimes e com divergência
naquele grupo. Para variáveis categóricas, uma pergunta como `polo_mp_Ativo <= 0.5`
quer dizer "não é recurso do MP".

**Pergunta.** Qual a primeira pergunta da árvore? Escreva, em linguagem natural,
a regra que leva à folha com a maior proporção de divergência.

[Volta ao Índice](#indice)

___
<div id="parte2"></div>

# 2. Escolher a profundidade

A profundidade é um **hiperparâmetro**. Complete a medida e rode: a célula calcula
a AUC no treino e na validação cruzada para cada profundidade.

In [ ]:
linhas = []
for prof in [1, 2, 3, 4, 5, 6, 8, 10, 12, 15]:
    modelo = make_pipeline(cria_preparo(), DecisionTreeClassifier(max_depth=prof, random_state=1))
    validacao = cross_val_score(modelo, X_treino, y_treino, cv=pastas, scoring="____").mean()
    modelo.fit(X_treino, y_treino)
    treino = roc_auc_score(y_treino, modelo.predict_proba(X_treino)[:, 1])
    linhas.append({"profundidade": prof, "treino": round(treino, 3), "validacao": round(validacao, 3)})
por_prof = pd.DataFrame(linhas)
por_prof

In [ ]:
(ggplot(por_prof.melt(id_vars="profundidade", var_name="base", value_name="auc"),
        aes(x="profundidade", y="auc", color="base"))
 + geom_line(size=1) + geom_point(size=3)
 + labs(x="profundidade máxima", y="AUC", color="")
 + theme_minimal())

**Perguntas.**

1. Qual a melhor profundidade pela validação cruzada?
2. O que acontece com a distância entre treino e validação quando a árvore
   cresce? Como isso se liga ao exercício 4 da aula 22?

[Volta ao Índice](#indice)

___
<div id="parte3"></div>

# 3. Floresta aleatória

A floresta faz a média de muitas árvores, cada uma com uma amostra sorteada dos
acórdãos e das variáveis. Complete o número de árvores com 200. A célula leva
cerca de um minuto.

In [ ]:
floresta = make_pipeline(cria_preparo(),
                         RandomForestClassifier(n_estimators=____, min_samples_leaf=3, n_jobs=-1, random_state=1))
auc_floresta = cross_val_score(floresta, X_treino, y_treino, cv=pastas, scoring="roc_auc").mean()
print("AUC na validação cruzada:", round(auc_floresta, 3))

A floresta não tem uma regra para ler, mas mostra quais variáveis mais ajudaram:
a **importância** soma, em todas as árvores, quanto cada variável melhorou as
divisões.

In [ ]:
floresta.fit(X_treino, y_treino)
nomes = [n.split("__")[-1] for n in floresta[0].get_feature_names_out()]
importancia = pd.Series(floresta[1].feature_importances_, index=nomes).sort_values(ascending=False)
importancia.head(10).round(3)

**Pergunta.** Por que a importância **não** quer dizer que a variável causa a
divergência? Pense na aula sobre confusão.

[Volta ao Índice](#indice)

___
<div id="parte4"></div>

# 4. Comparar os modelos

Com o mesmo treino e as mesmas pastas, a comparação é justa. Preencha o melhor
`C` da aula 23 (0,3) e a melhor profundidade que você achou na parte 2.

In [ ]:
modelos = {
    "logística": make_pipeline(cria_preparo(), LogisticRegression(C=____, max_iter=3000)),
    "árvore": make_pipeline(cria_preparo(), DecisionTreeClassifier(max_depth=____, random_state=1)),
    "floresta": make_pipeline(cria_preparo(), RandomForestClassifier(n_estimators=200, min_samples_leaf=3,
                                                                     n_jobs=-1, random_state=1)),
}
pd.Series({nome: cross_val_score(m, X_treino, y_treino, cv=pastas, scoring="roc_auc").mean()
           for nome, m in modelos.items()}).round(3)

**Pergunta.** Um tribunal quer usar o modelo para explicar aos gabinetes **por
que** um caso foi sinalizado. Outro só quer ordenar a fila de processos. Qual
modelo você indicaria para cada um? Justifique.

[Volta ao Índice](#indice)

___
<div id="parte5"></div>

# 5. O arquivo de predições: a receita do Projeto 08

No Projeto 08 você recebe `teste.csv`, que tem as mesmas colunas e **não tem a
resposta**. A receita tem três passos: ajustar o modelo escolhido em **todo o
treino**, prever no teste e salvar um CSV com `id` e `prob`.

Aqui o alvo é a divergência, só para treinar a receita. **Não suba este arquivo
no Projeto 08**: ele prevê outra coisa. Complete o método que dá as
probabilidades.

In [ ]:
teste = pd.read_csv("https://jtrecenti.github.io/cdad2-202662/ex08-competicao/dados/teste.csv")

modelo_final = modelos["floresta"]
modelo_final.fit(base[colunas], base["divergiu"])

predicoes = modelo_final.____(teste[colunas])[:, 1]

saida = pd.DataFrame({"id": teste["id"], "prob": predicoes})
saida.to_csv("predicoes_exemplo.csv", index=False)
saida.head()

### Do notebook ao Projeto 08

1. Troque o alvo: na primeira célula, use `base["negou"]` no lugar de
   `base["divergiu"]`.
2. Antes de usar uma coluna, pergunte: **no dia em que eu for prever, eu já sei
   isto?** A coluna `unanimidade` sai no mesmo julgamento que `negou`, mas está
   no `teste.csv`: decida com cuidado e justifique para você mesmo.
3. Compare modelos, variáveis e hiperparâmetros **por validação cruzada no
   treino**. Só depois use uma das 3 submissões.
4. No Colab, o arquivo baixa com
   `from google.colab import files; files.download("predicoes.csv")`.

**Pergunta.** O Projeto 08 limita a 3 submissões. O que aconteceria com a sua AUC
no teste, e com o significado dela, se as submissões fossem ilimitadas?

[Volta ao Índice](#indice)